# EgoDex 上的 ACT 数据缩放（Colab GPU）

线性 BC 只用手部状态，验证损失在大约 6.4 万个动作块之后就饱和了，全量也只比「保持不动」好大约 6%。这一本改用 **图像** 训练 lerobot **0.6.1** 的 ACT：16 步手腕增量，固定的 episode 验证集，三到四档数据量，再交给 `scripts/scaling_law.py`。

lerobot 0.6.1 的 ACT **没有语言编码器**。任务句子已经写在数据集的 `task` 字段里，命令不会假装打开语言条件。换到带语言头的策略时这些句子可以直接用。

数据是 [EgoDex test.zip](https://ml-site.cdn-apple.com/datasets/egodex/test.zip)（约 16.1GB，CC BY-NC-ND）。不要把 zip、HDF5 或导出数据集提交进仓库。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skyxiaobai/robot/blob/main/notebooks/egodex_act_scaling_colab.ipynb)

## 预期耗时

224 视频、ResNet18、`batch_size=4`。T4 上一步大约 0.4–1 秒，L4 可以试 `batch_size=8`。

| 模式 | 训练 episode | 优化步数 | 训练 | 加上下载、转换、224 导出 |
|---|---|---|---|---|
| 正式四档 | 64 / 256 / 1024 / 全部（约 1800） | 300 / 800 / 1500 / 2000 | 约 80 分钟 | 约 2.5–3.5 小时 |
| 只看方向 | 16 / 32 / 64（先 `--limit 80`） | 40 / 80 / 120 | 约 15 分钟 | 仍要下载 zip；转换和导出短很多 |

验证 episode 与 `ego_pretrain_bc.py` 一样：种子 0、留出 10%，各档共用。更小的档是训练 episode 顺序的前缀。


## 1. 运行时开关

`SUBSET = True` 只转换前 80 条，用来确认命令。正式曲线改成 `False`，并把 `SIZES` / `STEPS` 换成下面注释里的四档。`RUN_TRAIN` 在 GPU 运行时再打开。

In [ ]:
SUBSET = True
RUN_TRAIN = False  # 选 GPU 运行时再改成 True
LIMIT = 80 if SUBSET else 0
SIZES = "16,32,64" if SUBSET else "64,256,1024,all"
STEPS = "40,80,120" if SUBSET else "300,800,1500,2000"
BATCH_SIZE = 4
SEED = 0
DRIVE_DIR = "/content/drive/MyDrive/egodex"


## 2. 安装与仓库

训练 extra 带上 `lerobot-train` 需要的 accelerate。版本钉在 **0.6.1**。

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy", "h5py", "pyarrow", "pandas", "matplotlib"])
if RUN_TRAIN:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lerobot[training]==0.6.1"])

root = Path.cwd()
if not (root / "scripts" / "ego_act_scaling.py").exists():
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/skyxiaobai/robot.git", "/content/robot",
    ])
    os.chdir("/content/robot")
print("repo", Path.cwd())


## 3. 下载到 Google Drive

zip 约 16.1GB。子集模式也要这份 zip，但转换时用 `--limit`，不用把 3243 条都导出。已经下过就跳过。


In [ ]:
from google.colab import drive
import subprocess
from pathlib import Path

drive.mount("/content/drive")
drive_dir = Path(DRIVE_DIR)
drive_dir.mkdir(parents=True, exist_ok=True)
zip_path = drive_dir / "test.zip"
if not zip_path.is_file() or zip_path.stat().st_size < 17_000_000_000:
    subprocess.check_call([
        "curl", "-L", "--fail", "--retry", "3",
        "https://ml-site.cdn-apple.com/datasets/egodex/test.zip",
        "-o", str(zip_path),
    ])
extract_dir = Path("/content/egodex_test")
if not extract_dir.exists():
    extract_dir.mkdir(parents=True)
    subprocess.check_call(["unzip", "-q", str(zip_path), "-d", str(extract_dir)])
hdf5 = list(extract_dir.rglob("*.hdf5"))
print("hdf5", len(hdf5), "root", extract_dir)


## 4. 转换、质检、224 导出

`--video-size 224` 把真实 mp4 缩成正方形。动作每一行仍是下一步手腕增量；ACT 的 `chunk_size=16` 再拼成 16 步。

导出另有 `action_valid`（形状 `(2,)`，左手、右手）。当前帧和下一帧的手腕都是实测（有限数、不是补帧、不是丢掉的标注、置信度不是低于 0.5 的数字）才为 1。缺测仍写成「保持不动」的占位增量，但训练不算它。没有这一列的旧数据集会被当成全部有效，并打印警告。


In [ ]:
import subprocess
import sys
from pathlib import Path

source = Path("/content/egodex_test")
candidates = [source / "test", source]
source = next(path for path in candidates if path.is_dir() and any(path.rglob("*.hdf5")))
work = Path("/content/egodex_work")
unified = work / "unified"
yield_csv = work / "yield_episodes.csv"
dataset = work / "lerobot"
limit_args = ["--limit", str(LIMIT)] if LIMIT else []
subprocess.check_call([
    sys.executable, "scripts/convert_egodex.py",
    "--input", str(source), "--out", str(unified), "--workers", "2", *limit_args,
])
subprocess.check_call([
    sys.executable, "scripts/egodata_qc.py",
    "--episodes", str(unified),
    "--html", str(work / "yield_report.html"),
    "--csv", str(yield_csv),
])
subprocess.check_call([
    sys.executable, "scripts/ego_to_lerobot.py",
    "--episodes", str(unified),
    "--yield-csv", str(yield_csv),
    "--out", str(dataset),
    "--horizon", "16",
    "--video-size", "224",
])
print("dataset", dataset)


## 5. 四档 ACT

先 `--dry-run` 看命令和验证 episode。`RUN_TRAIN = True` 时才训练，并在固定验证集上写 `val_loss` 和 `copy_current_wrist`（都是动作 L1，只在 `action_valid=1` 的手上平均），然后出缩放报告。输出目录已存在时 lerobot 会拒绝覆盖，换一个 `OUT`。

训练命令进 `scripts/ego_act_train.py`：两只手都无效的步并进 lerobot 的 `action_is_pad`；只有一只手无效时用按维掩码，不把另一只手的实测增量一起丢掉。要跳过有效比例太低的块，给 `ego_act_scaling.py` 加 `--min-valid-fraction`（默认 0，块仍会抽到，但无效手不进损失）。


In [ ]:
import subprocess
import sys
from pathlib import Path

dataset = Path("/content/egodex_work/lerobot")
out = Path("/content/egodex_work/act_scaling")
common = [
    sys.executable, "scripts/ego_act_scaling.py",
    "--dataset", str(dataset),
    "--sizes", SIZES,
    "--steps", STEPS,
    "--batch-size", str(BATCH_SIZE),
    "--device", "cuda" if RUN_TRAIN else "cpu",
    "--seed", str(SEED),
    "--horizon", "16",
    "--out-dir", str(out),
]
if not RUN_TRAIN:
    subprocess.check_call(common + ["--dry-run"])
    print("未训练。GPU 运行时把 RUN_TRAIN 改成 True。")
else:
    subprocess.check_call(common + ["--run", "--max-eval-batches", "8"])
    print((out / "scaling_law.html").read_text(encoding="utf-8")[:500])


## 6. 本机 CPU 上确认命令能启动

没有 GPU 时用合成导出跑 **1 步**。占位视频是 16×16，只能说明 `lerobot-train` 和验证日志接得上，不能当成缩放曲线。Colab 里不必跑这一格。


In [ ]:
# 只在确认 CPU 安装了 lerobot 0.6.1 时手动跑。Colab GPU 流程不用这一格。
# python scripts/demo_open_dataset_pipeline.py --out /tmp/ego_cpu_demo
# python scripts/ego_to_lerobot.py --episodes /tmp/ego_cpu_demo/unified \
#     --yield-csv /tmp/ego_cpu_demo/yield_episodes.csv --out /tmp/ego_cpu_lerobot \
#     --horizon 16 --video-size 0
# python scripts/ego_act_scaling.py --dataset /tmp/ego_cpu_lerobot \
#     --sizes 1 --steps 1 --batch-size 2 --device cpu --horizon 16 \
#     --out-dir /tmp/ego_cpu_act --run --max-eval-batches 1
print("CPU 冒烟命令见上面的注释，以及 scripts/ego_act_scaling.py 的模块说明。")
